# Group 2 — Raw vs De-Identified Comparison + Two-Tier Age Model + Demo
Covers three tasks:

3. **Re-run the model-training pipeline on de-identified data**
4. **Build the raw-vs-de-identified comparison table**
5. **Wrap a two-tier age model and a demo function around this**

**Before running:**
1. Set `DATA_DIR` to your dataset folder (containing `db.csv`, `ecg/`, `ppg/`).
2. Make sure `results/features_deidentified.csv`, `results/labels_restricted.csv`, and
   `results/crosswalk_LOCAL_ONLY_DO_NOT_SHARE.csv` already exist (from the de-identification
   notebook) inside `DATA_DIR`.
3. Needs `scipy` and `scikit-learn`.

**On the crosswalk file:** this notebook reads it locally to align the train/test split between
the raw and de-identified runs (so both use the *same* subjects in each split, for a fair
comparison). It is read, never written out again, and must still not be shared outside your own
machine.


In [2]:
import json
import datetime
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.signal import find_peaks
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from numpy.random import default_rng

warnings.filterwarnings("ignore")

DATA_DIR = Path(r"C:\Users\User\mcd_rppg")   # e.g. Path(r"C:\Users\<you>\mcd_rppg")
DB_CSV = DATA_DIR / "db.csv"
RESULTS_DIR = DATA_DIR / "results"

SEED = 42
N_FOLDS = 5
TEST_FRAC = 0.2
AGE_ROUTING_THRESHOLD = 25   # two-tier split point

for p in [DB_CSV, RESULTS_DIR / "features_deidentified.csv",
          RESULTS_DIR / "labels_restricted.csv",
          RESULTS_DIR / "crosswalk_LOCAL_ONLY_DO_NOT_SHARE.csv"]:
    assert p.exists(), f"Missing required file: {p}"
print("All required files found.")


All required files found.


## 1. Rebuild the RAW feature table
Same extraction as the earlier training notebook, kept self-contained.


In [3]:
df = pd.read_csv(DB_CSV)
subj = df.drop_duplicates(["patient_id", "step"]).reset_index(drop=True)


def load_pw(path):
    vals, times = [], []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(None, 1)
            vals.append(float(parts[0]))
            times.append(parts[1])
    return np.array(vals, dtype=float), times


def ppg_sampling_rate(times):
    fmt = "%Y-%m-%d %H:%M:%S.%f"
    t0 = datetime.datetime.strptime(times[0], fmt)
    t1 = datetime.datetime.strptime(times[-1], fmt)
    duration = (t1 - t0).total_seconds()
    return (len(times) / duration if duration > 0 else None), duration


def hrv_features(signal, fs, min_bpm=40, max_bpm=180):
    if fs is None:
        return None
    min_dist = int(fs * 60 / max_bpm)
    peaks, _ = find_peaks(signal, distance=max(min_dist, 1),
                           prominence=(signal.max() - signal.min()) * 0.2)
    if len(peaks) < 4:
        return None
    ibi = np.diff(peaks) / fs
    ibi = ibi[(ibi >= 60/max_bpm) & (ibi <= 60/min_bpm)]
    if len(ibi) < 3:
        return None
    hr = 60 / ibi
    nn_diff = np.diff(ibi) * 1000
    pnn50 = np.mean(np.abs(nn_diff) > 50) * 100 if len(nn_diff) else np.nan
    return {"mean_hr": hr.mean(), "hr_std": hr.std(), "hr_range": hr.max() - hr.min(),
            "mean_ibi": ibi.mean(), "sdnn": ibi.std() * 1000,
            "rmssd": np.sqrt(np.mean(nn_diff**2)) if len(nn_diff) else np.nan,
            "pnn50": pnn50, "n_beats": len(peaks)}


def sdppg_features(signal, fs, min_bpm=40, max_bpm=180):
    if fs is None or len(signal) < 10:
        return None
    min_dist = int(fs * 60 / max_bpm)
    peaks, _ = find_peaks(signal, distance=max(min_dist, 1),
                           prominence=(signal.max() - signal.min()) * 0.2)
    if len(peaks) < 4:
        return None
    d2 = np.diff(signal, n=2)
    a_vals, b_vals = [], []
    for i in range(len(peaks) - 1):
        seg = d2[peaks[i]:peaks[i + 1]]
        if len(seg) < 5:
            continue
        half = seg[: max(len(seg) // 2, 3)]
        a, b = half.max(), half.min()
        if a > 0:
            a_vals.append(a)
            b_vals.append(b)
    if len(a_vals) < 3:
        return None
    a_vals, b_vals = np.array(a_vals), np.array(b_vals)
    return {"sdppg_a_mean": a_vals.mean(), "sdppg_b_mean": b_vals.mean(),
            "sdppg_ba_ratio": (b_vals / a_vals).mean(), "sdppg_n_beats": len(a_vals)}


FEATURE_COLS = ["mean_hr", "hr_std", "hr_range", "mean_ibi", "sdnn", "rmssd", "pnn50",
                 "sdppg_a_mean", "sdppg_b_mean", "sdppg_ba_ratio"]

rows = []
for _, r in subj.iterrows():
    pw_path = DATA_DIR / r.ppg
    row = {"patient_id": r.patient_id, "step": r.step, "sex": r.sex, "bmi": r.bmi,
           "age": r.age, "saturation": r.saturation}
    if pw_path.exists():
        try:
            vals, times = load_pw(pw_path)
            fs, dur = ppg_sampling_rate(times)
            hrv = hrv_features(vals, fs)
            sdp = sdppg_features(vals, fs)
            if hrv:
                row.update(hrv)
            if sdp:
                row.update(sdp)
        except Exception as e:
            print(f"  skipping {pw_path.name}: {e}")
    rows.append(row)

raw_features = pd.DataFrame(rows)
raw_features["sex_num"] = (raw_features["sex"] == "M").astype(int)
print(f"RAW: usable rows = {raw_features.dropna(subset=FEATURE_COLS).shape[0]} / {len(raw_features)}")


  skipping 8706_after.PW: time data '2023-10-' does not match format '%Y-%m-%d %H:%M:%S.%f'
RAW: usable rows = 1199 / 1200


## 2. Load the DE-IDENTIFIED feature and label files, and merge them back together
This is the actual "Task 3" starting point: the two files produced by the de-identification
notebook are joined back on `pseudonym_id` + `step`, exactly as that notebook's final section
described.


In [4]:
deid_features_raw = pd.read_csv(RESULTS_DIR / "features_deidentified.csv")
labels = pd.read_csv(RESULTS_DIR / "labels_restricted.csv")
crosswalk = pd.read_csv(RESULTS_DIR / "crosswalk_LOCAL_ONLY_DO_NOT_SHARE.csv")

deid_features = deid_features_raw.merge(labels, on=["pseudonym_id", "step"])
deid_features = deid_features.merge(crosswalk, on="pseudonym_id")  # attaches patient_id, for
                                                                     # split alignment ONLY
deid_features["sex_num"] = (deid_features["sex"] == "M").astype(int)

# One-hot encode bmi_bin, since exact bmi is no longer available post-generalisation
bmi_dummies = pd.get_dummies(deid_features["bmi_bin"], prefix="bmi_bin")
deid_features = pd.concat([deid_features, bmi_dummies], axis=1)
BMI_BIN_COLS = list(bmi_dummies.columns)

print(f"DE-IDENTIFIED: usable rows = {deid_features.dropna(subset=FEATURE_COLS).shape[0]} / {len(deid_features)}")
print(f"BMI bin categories found: {BMI_BIN_COLS}")


DE-IDENTIFIED: usable rows = 1199 / 1200
BMI bin categories found: ['bmi_bin_normal', 'bmi_bin_obese', 'bmi_bin_overweight', 'bmi_bin_suppressed', 'bmi_bin_underweight']


## 3. One shared subject-level split, used for BOTH raw and de-identified runs
Critical for a fair comparison: the same real subjects must be in the test set on both sides.
The split is computed once on `patient_id`, then applied to the raw table directly and to the
de-identified table via the crosswalk.


In [5]:
def subject_level_split(subject_ids, test_frac=TEST_FRAC, seed=SEED):
    ids = np.array(sorted(set(subject_ids)))
    rng = default_rng(seed)
    rng.shuffle(ids)
    n_test = int(len(ids) * test_frac)
    return set(ids[n_test:]), set(ids[:n_test])


train_ids, test_ids = subject_level_split(raw_features.patient_id.unique())
assert train_ids.isdisjoint(test_ids)
print(f"Train subjects: {len(train_ids)}  Test subjects: {len(test_ids)}  (shared by both runs)")

raw_train = raw_features[raw_features.patient_id.isin(train_ids)]
raw_test = raw_features[raw_features.patient_id.isin(test_ids)]
deid_train = deid_features[deid_features.patient_id.isin(train_ids)]
deid_test = deid_features[deid_features.patient_id.isin(test_ids)]
print(f"Raw:  train={len(raw_train)} test={len(raw_test)}")
print(f"Deid: train={len(deid_train)} test={len(deid_test)}")


Train subjects: 480  Test subjects: 120  (shared by both runs)
Raw:  train=960 test=240
Deid: train=960 test=240


## 4. Shared model-training helpers (same logic as the original pipeline)


In [6]:
def bootstrap_mae_ci(y_true, y_pred, n_boot=1000, seed=0):
    rng = default_rng(seed)
    n = len(y_true)
    errs = np.abs(np.asarray(y_true) - np.asarray(y_pred))
    boot_means = [errs[rng.integers(0, n, n)].mean() for _ in range(n_boot)]
    return np.mean(errs), np.percentile(boot_means, 2.5), np.percentile(boot_means, 97.5)


def cv_compare_models(data, target_col, demo_cols, feature_cols, n_folds=N_FOLDS, seed=SEED):
    data = data.dropna(subset=[target_col]).copy()
    groups = data["patient_id"].values
    gkf = GroupKFold(n_splits=n_folds)
    records = []
    for fold, (tr_idx, va_idx) in enumerate(gkf.split(data, groups=groups)):
        tr, va = data.iloc[tr_idx], data.iloc[va_idx]
        pred = np.full(len(va), tr[target_col].mean())
        records.append({"fold": fold, "model": "mean_predictor",
                         "mae": mean_absolute_error(va[target_col], pred)})
        for name, cols in [("demographics", demo_cols), ("feature", feature_cols)]:
            d_tr = tr.dropna(subset=cols + [target_col])
            d_va = va.dropna(subset=cols + [target_col])
            if len(d_tr) > 10 and len(d_va) > 3:
                scaler = StandardScaler().fit(d_tr[cols])
                model = RidgeCV(alphas=[0.01, 0.1, 1, 10, 100]).fit(
                    scaler.transform(d_tr[cols]), d_tr[target_col])
                pred = model.predict(scaler.transform(d_va[cols]))
                records.append({"fold": fold, "model": f"{name}_ridge",
                                 "mae": mean_absolute_error(d_va[target_col], pred)})
                if name == "feature" and len(d_tr) > 20:
                    rf = RandomForestRegressor(n_estimators=300, max_depth=5, random_state=seed,
                                                min_samples_leaf=5)
                    rf.fit(d_tr[cols], d_tr[target_col])
                    pred = rf.predict(d_va[cols])
                    records.append({"fold": fold, "model": "feature_random_forest",
                                     "mae": mean_absolute_error(d_va[target_col], pred)})
    return pd.DataFrame(records)


def best_model(cv_df):
    return cv_df.groupby("model")["mae"].mean().sort_values().index[0]


def fit_and_predict(train, test, target_col, cols, model_type, seed=SEED):
    tr = train.dropna(subset=cols + [target_col])
    te = test.dropna(subset=cols + [target_col])
    if len(tr) < 10 or len(te) < 3:
        return None, te
    if model_type == "mean_predictor":
        pred = np.full(len(te), tr[target_col].mean())
    elif model_type.endswith("_ridge"):
        scaler = StandardScaler().fit(tr[cols])
        model = RidgeCV(alphas=[0.01, 0.1, 1, 10, 100]).fit(scaler.transform(tr[cols]), tr[target_col])
        pred = model.predict(scaler.transform(te[cols]))
    elif model_type == "feature_random_forest":
        model = RandomForestRegressor(n_estimators=300, max_depth=5, random_state=seed,
                                       min_samples_leaf=5)
        model.fit(tr[cols], tr[target_col])
        pred = model.predict(te[cols])
    else:
        raise ValueError(model_type)
    return pred, te


## 5. Run the pipeline on RAW and on DE-IDENTIFIED data (Task 3)
For age: feature-based models use the same `FEATURE_COLS` on both sides (HRV/SDPPG features
are unaffected by de-identification, since de-identification only generalised the demographic
columns). For saturation: the demographics model uses exact `bmi` on the raw side and the
one-hot `bmi_bin` dummies on the de-identified side, since exact BMI is no longer available
post-generalisation.


In [7]:
AGE_DEMO_RAW = ["sex_num", "bmi"]
AGE_DEMO_DEID = ["sex_num"] + BMI_BIN_COLS
SAT_DEMO_RAW = ["sex_num", "age", "bmi"]
SAT_DEMO_DEID = ["sex_num", "age"] + BMI_BIN_COLS

results = {}
for label, train, test, age_demo, sat_demo in [
    ("raw", raw_train, raw_test, AGE_DEMO_RAW, SAT_DEMO_RAW),
    ("deidentified", deid_train, deid_test, AGE_DEMO_DEID, SAT_DEMO_DEID),
]:
    print(f"\n{'='*20} {label.upper()} {'='*20}")
    age_cv = cv_compare_models(train, "age", age_demo, FEATURE_COLS)
    age_best = best_model(age_cv) if len(age_cv) else "mean_predictor"
    age_cols = FEATURE_COLS if "feature" in age_best else age_demo
    age_pred, age_te = fit_and_predict(train, test, "age", age_cols, age_best)

    sat_cv = cv_compare_models(train, "saturation", sat_demo, FEATURE_COLS)
    sat_best = best_model(sat_cv) if len(sat_cv) else "mean_predictor"
    sat_cols = FEATURE_COLS if "feature" in sat_best else sat_demo
    sat_pred, sat_te = fit_and_predict(train, test, "saturation", sat_cols, sat_best)

    age_mae = bootstrap_mae_ci(age_te["age"], age_pred) if age_pred is not None else (None, None, None)
    sat_mae = bootstrap_mae_ci(sat_te["saturation"], sat_pred) if sat_pred is not None else (None, None, None)

    print(f"Age best model: {age_best}, test MAE: {age_mae[0]}")
    print(f"Saturation best model: {sat_best}, test MAE: {sat_mae[0]}")

    results[label] = {
        "age_best_model": age_best, "age_mae": age_mae, "age_pred": age_pred, "age_test": age_te,
        "sat_best_model": sat_best, "sat_mae": sat_mae, "sat_pred": sat_pred, "sat_test": sat_te,
        "age_cols": age_cols, "sat_cols": sat_cols,
    }



==================== RAW ====================
Age best model: feature_random_forest, test MAE: 3.87283845947172
Saturation best model: demographics_ridge, test MAE: 0.9126727466942853

==================== DEIDENTIFIED ====================
Age best model: feature_random_forest, test MAE: 3.87283845947172
Saturation best model: mean_predictor, test MAE: 0.9326388888888896


## 6. Build the raw-vs-de-identified comparison table (Task 4)
This is the direct answer to the project's core question: does performance survive
de-identification?


In [8]:
def stratified_mae(age_test, age_pred, threshold=AGE_ROUTING_THRESHOLD):
    if age_pred is None:
        return None, None
    err = np.abs(age_test["age"].values - age_pred)
    young = err[age_test["age"].values < threshold]
    old = err[age_test["age"].values >= threshold]
    return (young.mean() if len(young) else None), (old.mean() if len(old) else None)


comparison_rows = []
for label in ["raw", "deidentified"]:
    r = results[label]
    young_mae, old_mae = stratified_mae(r["age_test"], r["age_pred"])
    comparison_rows.append({
        "dataset": label,
        "age_model": r["age_best_model"], "age_overall_mae": r["age_mae"][0],
        "age_under25_mae": young_mae, "age_25plus_mae": old_mae,
        "sat_model": r["sat_best_model"], "sat_mae": r["sat_mae"][0],
    })

comparison = pd.DataFrame(comparison_rows)
print(comparison.to_string(index=False))

comparison_path = RESULTS_DIR / "raw_vs_deidentified_comparison.csv"
comparison.to_csv(comparison_path, index=False)
print(f"\nSaved: {comparison_path}")


     dataset             age_model  age_overall_mae  age_under25_mae  age_25plus_mae          sat_model  sat_mae
         raw feature_random_forest         3.872838         2.769317       11.597486 demographics_ridge 0.912673
deidentified feature_random_forest         3.872838         2.769317       11.597486     mean_predictor 0.932639

Saved: C:\Users\User\mcd_rppg\results\raw_vs_deidentified_comparison.csv


## 7. Two-tier age model (Task 5, part 1)
A two-stage cascade: a full-population model gives an initial age estimate, which routes the
subject to a young-specialist or old-specialist model for the final prediction. Built and
evaluated on **both** raw and de-identified data, using the same routing threshold.


In [9]:
def build_two_tier_age_model(train, feature_cols, threshold=AGE_ROUTING_THRESHOLD, seed=SEED):
    tr = train.dropna(subset=feature_cols + ["age"])
    router = RandomForestRegressor(n_estimators=300, max_depth=5, random_state=seed,
                                    min_samples_leaf=5)
    router.fit(tr[feature_cols], tr["age"])

    young_tr = tr[tr["age"] < threshold]
    old_tr = tr[tr["age"] >= threshold]

    def _fit_or_fallback(subset):
        if len(subset) >= 20:
            m = RandomForestRegressor(n_estimators=300, max_depth=4, random_state=seed,
                                       min_samples_leaf=3)
            m.fit(subset[feature_cols], subset["age"])
            return m
        return None  # too few subjects — fall back to the router's own estimate

    young_model = _fit_or_fallback(young_tr)
    old_model = _fit_or_fallback(old_tr)
    print(f"  Two-tier models trained on {len(young_tr)} young / {len(old_tr)} old subjects "
          f"({'specialist' if old_model else 'FALLBACK to router — too few old subjects'} for 25+)")
    return router, young_model, old_model


def predict_two_tier(router, young_model, old_model, test, feature_cols,
                      threshold=AGE_ROUTING_THRESHOLD):
    te = test.dropna(subset=feature_cols)
    initial = router.predict(te[feature_cols])
    final = initial.copy()
    young_mask = initial < threshold
    if young_model is not None and young_mask.any():
        final[young_mask] = young_model.predict(te.loc[young_mask, feature_cols])
    if old_model is not None and (~young_mask).any():
        final[~young_mask] = old_model.predict(te.loc[~young_mask, feature_cols])
    return final, te


two_tier_rows = []
for label, train, test in [("raw", raw_train, raw_test), ("deidentified", deid_train, deid_test)]:
    print(f"\n{label}:")
    router, young_m, old_m = build_two_tier_age_model(train, FEATURE_COLS)
    pred, te = predict_two_tier(router, young_m, old_m, test, FEATURE_COLS)
    young_mae, old_mae = stratified_mae(te, pred)
    overall_mae, lo, hi = bootstrap_mae_ci(te["age"], pred)
    print(f"  Two-tier — overall: {overall_mae:.2f}, under-25: {young_mae}, 25+: {old_mae}")
    two_tier_rows.append({"dataset": label, "two_tier_overall_mae": overall_mae,
                            "two_tier_under25_mae": young_mae, "two_tier_25plus_mae": old_mae})

two_tier_comparison = pd.DataFrame(two_tier_rows)
print()
print(two_tier_comparison.to_string(index=False))
two_tier_comparison.to_csv(RESULTS_DIR / "two_tier_age_comparison.csv", index=False)



raw:
  Two-tier models trained on 879 young / 80 old subjects (specialist for 25+)
  Two-tier — overall: 4.13, under-25: 3.5984825092373587, 25+: 7.855497066833847

deidentified:
  Two-tier models trained on 879 young / 80 old subjects (specialist for 25+)
  Two-tier — overall: 4.13, under-25: 3.5984825092373587, 25+: 7.855497066833847

     dataset  two_tier_overall_mae  two_tier_under25_mae  two_tier_25plus_mae
         raw              4.130609              3.598483             7.855497
deidentified              4.130609              3.598483             7.855497


## 8. Demo function (Task 5, part 2)
Takes one subject's `.PW` file plus their sex and BMI, and returns a predicted age (via the
two-tier model) and a predicted saturation. Works entirely from the signal for age; saturation
needs sex/BMI because that was the winning model. A simple out-of-distribution flag is included,
since a number alone isn't a useful clinical output.


In [10]:
def demo_predict(pw_path, sex, bmi, router, young_model, old_model, sat_model, sat_cols,
                  train_reference, feature_cols=FEATURE_COLS, threshold=AGE_ROUTING_THRESHOLD):
    '''
    pw_path: path to a .PW file
    sex: "M" or "F"
    bmi: a numeric BMI value (exact; this demo runs on raw-style input)
    sat_model: a fitted model object (RidgeCV, or None to use the training mean) for saturation
    sat_cols: the feature columns sat_model expects, in order
    train_reference: the training DataFrame used to build the feature-range OOD check
    '''
    vals, times = load_pw(pw_path)
    fs, dur = ppg_sampling_rate(times)
    hrv = hrv_features(vals, fs) or {}
    sdp = sdppg_features(vals, fs) or {}
    feats = {**hrv, **sdp}

    missing = [c for c in feature_cols if c not in feats or pd.isna(feats[c])]
    if missing:
        return {"error": f"Could not extract required features: {missing}. "
                          f"Recording may be too short or noisy."}

    x = pd.DataFrame([feats])[feature_cols]

    # out-of-distribution check: is this recording's features within the training range?
    ood_flags = []
    for col in feature_cols:
        lo, hi = train_reference[col].min(), train_reference[col].max()
        if not (lo <= x[col].iloc[0] <= hi):
            ood_flags.append(col)

    initial_age = router.predict(x)[0]
    if initial_age < threshold and young_model is not None:
        age_pred = young_model.predict(x)[0]
    elif initial_age >= threshold and old_model is not None:
        age_pred = old_model.predict(x)[0]
    else:
        age_pred = initial_age  # fallback to the router's own estimate

    sex_num = 1 if sex == "M" else 0
    sat_input = {"sex_num": sex_num, "age": age_pred, "bmi": bmi}
    sat_row = pd.DataFrame([sat_input])
    sat_row = sat_row.reindex(columns=sat_cols, fill_value=0)  # handles bmi_bin dummy columns
    sat_pred = sat_model.predict(sat_row)[0] if sat_model is not None else np.nan

    confidence = "low" if ood_flags else "normal"
    return {
        "predicted_age": round(float(age_pred), 1),
        "predicted_saturation": round(float(sat_pred), 1),
        "confidence": confidence,
        "out_of_distribution_features": ood_flags,
    }


print("demo_predict() defined. Example usage (edit the path/sex/bmi for a real subject):")
print('''
  from sklearn.linear_model import RidgeCV
  router, young_m, old_m = build_two_tier_age_model(raw_train, FEATURE_COLS)
  scaler = StandardScaler().fit(raw_train.dropna(subset=SAT_DEMO_RAW+["saturation"])[SAT_DEMO_RAW])
  sat_model = RidgeCV().fit(
      scaler.transform(raw_train.dropna(subset=SAT_DEMO_RAW+["saturation"])[SAT_DEMO_RAW]),
      raw_train.dropna(subset=SAT_DEMO_RAW+["saturation"])["saturation"])
  result = demo_predict(DATA_DIR / "ppg/1020_after.PW", sex="F", bmi=19.0,
                         router=router, young_model=young_m, old_model=old_m,
                         sat_model=sat_model, sat_cols=SAT_DEMO_RAW, train_reference=raw_train)
  print(result)
''')


demo_predict() defined. Example usage (edit the path/sex/bmi for a real subject):

  from sklearn.linear_model import RidgeCV
  router, young_m, old_m = build_two_tier_age_model(raw_train, FEATURE_COLS)
  scaler = StandardScaler().fit(raw_train.dropna(subset=SAT_DEMO_RAW+["saturation"])[SAT_DEMO_RAW])
  sat_model = RidgeCV().fit(
      scaler.transform(raw_train.dropna(subset=SAT_DEMO_RAW+["saturation"])[SAT_DEMO_RAW]),
      raw_train.dropna(subset=SAT_DEMO_RAW+["saturation"])["saturation"])
  result = demo_predict(DATA_DIR / "ppg/1020_after.PW", sex="F", bmi=19.0,
                         router=router, young_model=young_m, old_model=old_m,
                         sat_model=sat_model, sat_cols=SAT_DEMO_RAW, train_reference=raw_train)
  print(result)



In [13]:
list((DATA_DIR / "ppg").glob("*.PW"))

[WindowsPath('C:/Users/User/mcd_rppg/ppg/1020_after.PW'),
 WindowsPath('C:/Users/User/mcd_rppg/ppg/1020_before.PW'),
 WindowsPath('C:/Users/User/mcd_rppg/ppg/1024_after.PW'),
 WindowsPath('C:/Users/User/mcd_rppg/ppg/1024_before.PW'),
 WindowsPath('C:/Users/User/mcd_rppg/ppg/1035_after.PW'),
 WindowsPath('C:/Users/User/mcd_rppg/ppg/1035_before.PW'),
 WindowsPath('C:/Users/User/mcd_rppg/ppg/1091_after.PW'),
 WindowsPath('C:/Users/User/mcd_rppg/ppg/1091_before.PW'),
 WindowsPath('C:/Users/User/mcd_rppg/ppg/1097_after.PW'),
 WindowsPath('C:/Users/User/mcd_rppg/ppg/1097_before.PW'),
 WindowsPath('C:/Users/User/mcd_rppg/ppg/1099_after.PW'),
 WindowsPath('C:/Users/User/mcd_rppg/ppg/1099_before.PW'),
 WindowsPath('C:/Users/User/mcd_rppg/ppg/1107_after.PW'),
 WindowsPath('C:/Users/User/mcd_rppg/ppg/1107_before.PW'),
 WindowsPath('C:/Users/User/mcd_rppg/ppg/1113_after.PW'),
 WindowsPath('C:/Users/User/mcd_rppg/ppg/1113_before.PW'),
 WindowsPath('C:/Users/User/mcd_rppg/ppg/1115_after.PW'),
 Windo

In [18]:
router, young_m, old_m = build_two_tier_age_model(raw_train, FEATURE_COLS)
scaler = StandardScaler().fit(raw_train.dropna(subset=SAT_DEMO_RAW+["saturation"])[SAT_DEMO_RAW])
sat_model = RidgeCV().fit(
    scaler.transform(raw_train.dropna(subset=SAT_DEMO_RAW+["saturation"])[SAT_DEMO_RAW]),
    raw_train.dropna(subset=SAT_DEMO_RAW+["saturation"])["saturation"])
result = demo_predict(DATA_DIR / "ppg/3930_before.PW", sex="F", bmi=19.0,
                       router=router, young_model=young_m, old_model=old_m,
                       sat_model=sat_model, sat_cols=SAT_DEMO_RAW, train_reference=raw_train)
print(result)

  Two-tier models trained on 879 young / 80 old subjects (specialist for 25+)
{'predicted_age': 64.5, 'predicted_saturation': 80.7, 'confidence': 'normal', 'out_of_distribution_features': []}


In [19]:
patient_id = 1266   # the number from the filename you picked, before the first underscore
df[df.patient_id == patient_id][["sex", "bmi"]].drop_duplicates()

,sex,bmi
108,M,23.80869


In [20]:
df[df.patient_id == 3930][["sex", "bmi", "age", "saturation", "step"]].drop_duplicates()

,sex,bmi,age,saturation,step
1206,F,30.407627,73.0,95.0,after
1209,F,30.407627,73.0,96.0,before


In [21]:
result = demo_predict(DATA_DIR / "ppg/3930_before.PW", sex="F", bmi=30.4,
                       router=router, young_model=young_m, old_model=old_m,
                       sat_model=sat_model, sat_cols=SAT_DEMO_RAW, train_reference=raw_train)
print(result)

{'predicted_age': 64.5, 'predicted_saturation': 79.5, 'confidence': 'normal', 'out_of_distribution_features': []}


In [23]:
for pid, fname in [(3930, "3930_before.PW"), (1386, "1386_after.PW")]:
    row = df[df.patient_id == pid][["sex","bmi","age","saturation"]].drop_duplicates().iloc[0]
    result = demo_predict(DATA_DIR / f"ppg/{fname}", sex=row.sex, bmi=row.bmi,
                           router=router, young_model=young_m, old_model=old_m,
                           sat_model=sat_model, sat_cols=SAT_DEMO_RAW, train_reference=raw_train)
    print(pid, "actual age:", row.age, "actual sat:", row.saturation, "-->", result)

3930 actual age: 73.0 actual sat: 95.0 --> {'predicted_age': 64.5, 'predicted_saturation': 79.5, 'confidence': 'normal', 'out_of_distribution_features': []}
1386 actual age: 19.0 actual sat: 99.0 --> {'predicted_age': 19.9, 'predicted_saturation': 89.9, 'confidence': 'normal', 'out_of_distribution_features': []}


In [24]:
# "Leaky" baseline: saturation MAE using TRUE age (what you've reported so far)
true_age_input = raw_test[["sex_num", "age", "bmi"]].dropna()
pred_leaky = sat_model.predict(scaler.transform(true_age_input))
mae_leaky, _, _ = bootstrap_mae_ci(raw_test.loc[true_age_input.index, "saturation"], pred_leaky)

# Realistic end-to-end: saturation MAE using PREDICTED age (what the demo actually does)
te = raw_test.dropna(subset=FEATURE_COLS + ["sex_num", "bmi", "saturation"])
initial = router.predict(te[FEATURE_COLS])
age_chain = np.where(initial < AGE_ROUTING_THRESHOLD,
                      young_m.predict(te[FEATURE_COLS]) if young_m else initial,
                      old_m.predict(te[FEATURE_COLS]) if old_m else initial)
chained_input = pd.DataFrame({"sex_num": te["sex_num"].values, "age": age_chain, "bmi": te["bmi"].values})
pred_chained = sat_model.predict(scaler.transform(chained_input))
mae_chained, _, _ = bootstrap_mae_ci(te["saturation"], pred_chained)

print(f"Saturation MAE using TRUE age (reported benchmark):      {mae_leaky:.2f}")
print(f"Saturation MAE using PREDICTED age (real deployment):    {mae_chained:.2f}")

Saturation MAE using TRUE age (reported benchmark):      0.91
Saturation MAE using PREDICTED age (real deployment):    0.93


## 9. Summary checklist

In [25]:
print("=" * 60)
print("SUMMARY CHECKLIST")
print("=" * 60)
checklist = [
    "[ ] Re-ran the pipeline on de-identified data using the SAME train/test subjects as raw",
    "[ ] Raw-vs-de-identified comparison table built and saved",
    "[ ] Two-tier age model built and evaluated on both raw and de-identified data",
    "[ ] Demo function tested on at least one real .PW file",
    "[ ] Checked whether the old-subject specialist model had enough data, or fell back to the router",
    "[ ] Compared de-identified MAE against raw MAE — is the project's core question answered?",
]
for item in checklist:
    print(item)


SUMMARY CHECKLIST
[ ] Re-ran the pipeline on de-identified data using the SAME train/test subjects as raw
[ ] Raw-vs-de-identified comparison table built and saved
[ ] Two-tier age model built and evaluated on both raw and de-identified data
[ ] Demo function tested on at least one real .PW file
[ ] Checked whether the old-subject specialist model had enough data, or fell back to the router
[ ] Compared de-identified MAE against raw MAE — is the project's core question answered?
